# Lecture 3 — Class Exercise
## Line Charts: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
- No spaghetti — if you show multiple lines, grey everything and highlight one
- Remove chart borders, heavy gridlines, rotating text
- Label directly; avoid legends where possible
- Insight title


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('../data/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [2]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


## Task 1 - A line chart showing CO2 per Capita for all countries in the dataset

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#EEEEEE), thinner
- Highlighted country/countries **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country/countries and its story

In [1]:
# Task 1 — highlight the country with the largest % change in CO2 per capita

pc = df.pivot(index='Country', columns='Year', values='CO2_per_capita')
pct = ((pc[2022] / pc[2000] - 1) * 100).round(1)
country = pct.abs().idxmax()  # China: steepest rise
start, end = pc.loc[country, [2000, 2022]]

data = df.assign(group=df['Country'].where(df['Country'] == country, 'Other'))

fig = px.line(
    data_frame=data,
    x='Year',
    y='CO2_per_capita',
    color='group',
    line_group='Country',
    color_discrete_map={country: '#2E75B6', 'Other': '#EEEEEE'},
    hover_name='Country',
    hover_data={'group': False},
    labels={'CO2_per_capita': 'CO₂ per Capita (tonnes)'},
    template='plotly_white',
    title=(
        f"{country}'s CO₂ per capita rose {abs(pct[country]):.0f}% "
        f"between 2000 and 2022 — the largest shift in the set"
    ),
    height=420,
)

fig.update_traces(line_width=1)
fig.update_traces(line_width=3, selector={'name': country})
fig.add_annotation(
    x=2022,
    y=end,
    text=country,
    showarrow=False,
    xanchor='left',
    xshift=6,
    font=dict(color='#2E75B6', size=13),
)
fig.update_layout(
    showlegend=False,
    yaxis=dict(showgrid=False),
    xaxis=dict(showgrid=False, title=''),
    font=dict(family='Arial', size=13),
)
fig.show()

---
## Task 2 — A line chart showing CO2 emissions over time for all Asian countries in the dataset, with one country highlighted.

**same requirements as Task1**

In [2]:
# Task 2 — Asia only: highlight the fastest-growing emitter since 2000

asia = df[df['Region'] == 'Asia'].copy()
mt = asia.pivot(index='Country', columns='Year', values='CO2_Mt')
country = (mt[2022] / mt[2000]).idxmax()  # China
start, end = mt.loc[country, [2000, 2022]]

data = asia.assign(group=asia['Country'].where(asia['Country'] == country, 'Other'))

fig = px.line(
    data_frame=data,
    x='Year',
    y='CO2_Mt',
    color='group',
    line_group='Country',
    color_discrete_map={country: '#2E75B6', 'Other': '#EEEEEE'},
    hover_name='Country',
    hover_data={'group': False},
    labels={'CO2_Mt': 'CO₂ emissions (million tonnes)'},
    template='plotly_white',
    title=(
        f"{country}'s total CO₂ emissions grew {end / start:.1f}× since 2000 "
        f"— the fastest rise among Asian countries in this dataset"
    ),
    height=420,
)

fig.update_traces(line_width=1)
fig.update_traces(line_width=3, selector={'name': country})
fig.add_annotation(
    x=2022,
    y=end,
    text=country,
    showarrow=False,
    xanchor='left',
    xshift=6,
    font=dict(color='#2E75B6', size=13),
)
fig.update_layout(
    showlegend=False,
    yaxis=dict(showgrid=False),
    xaxis=dict(showgrid=False, title=''),
    font=dict(family='Arial', size=13),
)
fig.show()
